# advanced_training

Generated with **Lispyn** — originally created by **Eduardo J. Barrios**.

Source: [Lispyn](https://github.com/edujbarrios/lispyn) (Apache-2.0).

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='advanced_training', assignments=[['md', 'md']], prefix='')


# Advanced training with Lispyn

Use gradient accumulation, global gradient clipping, and a cosine learning-rate scheduler in a real CPU PyTorch experiment. This tiny classification dataset is synthetic; training results are real and deterministic when run.

In [ ]:
import hy
import torch
torch.manual_seed(21)
model = torch.nn.Sequential(torch.nn.Linear(2, 12), torch.nn.ReLU(), torch.nn.Linear(12, 2))
x = torch.tensor([[0.0, 0.0], [0.2, 0.1], [0.9, 0.1], [0.8, 0.2], [0.1, 0.9], [0.2, 0.8], [1.0, 1.0], [0.9, 0.8], [0.0, 0.2], [0.9, 0.0], [0.0, 1.0], [1.0, 0.9]])
y = torch.tensor([0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1])
training = torch.utils.data.TensorDataset(x[slice(0, 8)], y[slice(0, 8)])
validation = torch.utils.data.TensorDataset(x[slice(8, None)], y[slice(8, None)])


## Accumulate gradients and schedule the learning rate

The train-accumulating Hy macro keeps native PyTorch optimizers. Three microbatches contribute to each optimizer update. Gradient norms are clipped at 1.0, and the cosine scheduler advances once per epoch.

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='advanced_training', assignments=[['train-accumulating', 'train-accumulating']], prefix='')
from lispyn.workflow import train
run = train(model, training, accumulation_steps=3, validation=validation, loss_fn=torch.nn.CrossEntropyLoss(), metrics='accuracy', epochs=12, batch_size=1, lr=0.025, gradient_clip=1.0, scheduler='cosine', scheduler_kwargs={'eta_min': 0.001}, device='cpu', seed=21, experiment_dir='runs/advanced-training')
print('Epochs:', len(run.history['train_loss']))
print('Recorded learning rates:', run.history['learning_rate'])


## Plot training and learning-rate curves

The plots share a Docusaurus-inspired dark background and a reusable turquoise palette. Points come from an actual offline training run, not fabricated training results.

In [ ]:
import hy
hy.macros.require('lispyn.macros', None, target_module_name='advanced_training', assignments=[['defcurve', 'defcurve'], ['learning-rate-curve', 'learning-rate-curve']], prefix='')
from lispyn.plots import make_plotter
dark_curves = make_plotter(theme='docusaurus-dark', palette='turquoise-spectrum', train_color='#40E0D0', val_color='#8CF0EB')
dark_curves(run, metric='loss', title='Accumulated PyTorch training', subtitle='12 epochs · CPU · 3 microbatches/update', save='runs/advanced-training/loss.png')
from lispyn.plots import plot_learning_rate
plot_learning_rate(run, theme='docusaurus-dark', palette='turquoise-spectrum', title='Cosine learning-rate schedule', subtitle='Recorded optimizer LR at each epoch', save='runs/advanced-training/learning_rate.png')
